## Experiment 3 — Three-Way Triplet Fine-Tuning


**Main Question:** Can explicitly teaching PRO, CON, and NEUTRAL improve three-way leakage-direction classification compared with directional-only fine-tuning?


## Setup

Install dependencies, authenticate Colab, and open the source spreadsheet.


In [1]:
# SETUP — Install the packages required by this independent Colab notebook.
# The version-free installs preserve the historical notebook behavior while allowing a fresh runtime to resolve compatible packages.
!pip install -q sentence-transformers datasets accelerate gspread pandas scikit-learn


In [2]:
# SETUP 2 — Authenticate Colab and connect to Google Sheets.
# The authenticated gspread client is used only to read the frozen dataset tabs in this notebook.
from google.colab import auth
auth.authenticate_user()

import google.auth
import gspread
import pandas as pd
import numpy as np

creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Import display explicitly so table previews also work outside Colab's implicit namespace.
from IPython.display import display


In [3]:
# SETUP 3 — Open the ballot-leakage spreadsheet by URL.
# Listing worksheet titles confirms that TRAIN, VALIDATION, TEST, and summary tabs are available.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1HnVdDmYMZnnEDZANBYWM3RJ0w7WEpXQGZrCxWkyoqMQ/edit"

spreadsheet = gc.open_by_url(SHEET_URL)

print([ws.title for ws in spreadsheet.worksheets()])


['Candidate Dataset', 'Source Ballots', 'Read Me', 'TRAIN', 'VALIDATION', 'TEST', 'SPLIT SUMMARY', 'BGE BASELINE VALIDATION', 'BGE BASELINE METRICS']


## Load Dataset

Load and normalize the frozen TRAIN and VALIDATION tabs without changing annotations.


In [4]:
# Cell 37 — Load TRAIN
# STEP 1 — Read the frozen training split and standardize its annotation fields.
# sentence is the model input; judge_id and ballot_id remain for traceability and overlap checks.
train_ws = spreadsheet.worksheet("TRAIN")
train_df = pd.DataFrame(
    train_ws.get_all_records()
)

# Convert strength labels to numbers and make direction labels consistent.
train_df["leakage_strength"] = pd.to_numeric(
    train_df["leakage_strength"]
)
train_df["leakage_direction"] = (
    train_df["leakage_direction"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Confirm the loaded size and class distributions before training.
print("TRAIN shape:", train_df.shape)
print(train_df["leakage_strength"].value_counts())
print(train_df["leakage_direction"].value_counts())


TRAIN shape: (285, 15)
leakage_strength
0    128
1    103
2     54
Name: count, dtype: int64
leakage_direction
NEUTRAL    128
CON         93
PRO         64
Name: count, dtype: int64


In [5]:
# BASELINE 1 — Load the frozen VALIDATION tab into a pandas DataFrame.
# These 44 rows are used for evaluation and threshold selection, never for updating model weights.
validation_ws = spreadsheet.worksheet("VALIDATION")

validation_df = pd.DataFrame(
    validation_ws.get_all_records()
)

print(validation_df.shape)
validation_df.head()


(61, 15)


,sentence_id,ballot_id,judge_id,actual_ballot,section,sentence,previous_sentence,next_sentence,leakage_strength,leakage_direction,needs_context,annotation_uncertain,uncertainty_reason,data_source,ballot_record_id
0,FA_R1_B2-304_S0016,FA_R1_B2-304,Hailley Yang,CON,Comments to CON,Needs more direct refutation of PRO’s economic...,The logic between economic paralysis and colla...,CON 2 : Good at picking apart CON’s examples; ...,1,PRO,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
1,FA_R1_B2-304_S0030,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,"Con2 also claims that STEMS are wasting money,...",Some contentions are not that relevant to the ...,Con2 is also good at pointing out that Pro can...,1,PRO,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
2,FA_R1_B2-304_S0031,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,Con2 is also good at pointing out that Pro can...,"Con2 also claims that STEMS are wasting money,...",Pro 1 can give more evidence about why exactly...,1,CON,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
3,FA_R1_B2-304_S0033,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,Con also points out that proponents fail to gi...,Pro 1 can give more evidence about why exactly...,"Overall speaking, con made more contributions ...",1,CON,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
4,FA_R1_B2-304_S0034,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,"Overall speaking, con made more contributions ...",Con also points out that proponents fail to gi...,,2,CON,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304


In [6]:
# BASELINE 2 — Standardize VALIDATION labels and inspect their distributions.
# Numeric strength and normalized direction values prevent type or capitalization errors in later metrics.
validation_df["leakage_strength"] = pd.to_numeric(
    validation_df["leakage_strength"]
)

validation_df["leakage_direction"] = (
    validation_df["leakage_direction"]
    .astype(str)
    .str.upper()
    .str.strip()
)

print(validation_df["leakage_strength"].value_counts())
print()
print(validation_df["leakage_direction"].value_counts())
print()
print(validation_df["section"].value_counts())


leakage_strength
0    28
1    21
2    12
Name: count, dtype: int64

leakage_direction
NEUTRAL    28
CON        19
PRO        14
Name: count, dtype: int64

section
Reason for Decision    34
Comments to CON        14
Comments to PRO        13
Name: count, dtype: int64


## Verify Split

Confirm that judge and ballot identities remain disjoint between TRAIN and VALIDATION.


In [7]:
# VERIFY SPLIT — Confirm the frozen TRAIN and VALIDATION groups do not overlap.
# TEST is deliberately not loaded, scored, sampled, or inspected anywhere in this notebook.
required_columns = {
    "sentence_id", "ballot_id", "judge_id", "sentence",
    "leakage_strength", "leakage_direction", "section"
}
assert required_columns.issubset(train_df.columns), "TRAIN is missing required columns"
assert required_columns.issubset(validation_df.columns), "VALIDATION is missing required columns"

train_judges = set(train_df["judge_id"].astype(str))
validation_judges = set(validation_df["judge_id"].astype(str))
train_ballots = set(train_df["ballot_id"].astype(str))
validation_ballots = set(validation_df["ballot_id"].astype(str))

assert train_judges.isdisjoint(validation_judges), "TRAIN/VALIDATION judge overlap"
assert train_ballots.isdisjoint(validation_ballots), "TRAIN/VALIDATION ballot overlap"

print("TRAIN/VALIDATION judge overlap:", len(train_judges & validation_judges))
print("TRAIN/VALIDATION ballot overlap:", len(train_ballots & validation_ballots))
print("TEST remains untouched and is not loaded.")


TRAIN/VALIDATION judge overlap: 0
TRAIN/VALIDATION ballot overlap: 0
TEST remains untouched and is not loaded.


## Model / Targets

Create the Experiment 2 distribution and the original symmetric PRO/CON/NEUTRAL target banks.


### Training Subset

All available directional TRAIN examples are retained (31 PRO and 52 CON). Since TRAIN contains 129 NEUTRAL examples, Experiment 2 uses a fixed random sample of 60 NEUTRAL sentences to introduce explicit three-way training without allowing NEUTRAL to dominate the training set.

Experiment 3 will remove this sampling decision and repeat the same method using the full TRAIN distribution.

In [8]:
# STEP 2 — Build the Experiment 2 training distribution.
# Keep only the three annotated direction classes.
training_df = train_df[
    train_df["leakage_direction"].isin(
        ["PRO", "CON", "NEUTRAL"]
    )
].copy()

# Inspect the full class distribution before any balancing.
print("Full TRAIN examples:", len(training_df))
print(
    training_df["leakage_direction"]
    .value_counts()
)

# Keep all PRO and CON rows, but sample 60 NEUTRAL rows so the majority class does not dominate Experiment 2.
pro_df = training_df[
    training_df["leakage_direction"] == "PRO"
]
con_df = training_df[
    training_df["leakage_direction"] == "CON"
]
neutral_df = training_df[
    training_df["leakage_direction"] == "NEUTRAL"
].sample(
    n=60,
    random_state=42
)

# Combine the classes and shuffle them reproducibly.
balanced_train_df = pd.concat(
    [pro_df, con_df, neutral_df],
    ignore_index=True
)
balanced_train_df = (
    balanced_train_df
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Confirm the exact distribution that will produce training triplets.
print()
print("Fine-tuning distribution:")
print(
    balanced_train_df["leakage_direction"]
    .value_counts()
)


Full TRAIN examples: 285
leakage_direction
NEUTRAL    128
CON         93
PRO         64
Name: count, dtype: int64

Fine-tuning distribution:
leakage_direction
CON        93
PRO        64
NEUTRAL    60
Name: count, dtype: int64


### Triplet Construction

Each training sentence is used as an anchor. A target from its correct class (PRO, CON, or NEUTRAL) is used as the positive, while targets from each of the two incorrect classes are used as negatives.

With four target formulations and two incorrect classes, each source sentence is reused in 8 triplet comparisons. These are repeated training comparisons, not additional independent sentences.

In [9]:
# STEP 3 — Define symmetric semantic target banks.
# These short prompts represent PRO, CON, and NEUTRAL meanings; they are comparison anchors, not extra ballot examples.
# Cell 39 — Define simple symmetric targets
# Three-way semantic targets
pro_targets = [
    "Pro wins the debate.",
    "The judge favors Pro over Con.",
    "Pro has the stronger argument than Con.",
    "Con fails against Pro's argument."
]
con_targets = [
    "Con wins the debate.",
    "The judge favors Con over Pro.",
    "Con has the stronger argument than Pro.",
    "Pro fails against Con's argument."
]
neutral_targets = [
    "This feedback does not indicate which side won.",
    "Neither side is clearly favored by this feedback.",
    "This is general feedback rather than a ballot decision.",
    "The sentence does not reveal the judge's vote."
]


In [10]:
# STEP 4 — Convert each labeled TRAIN sentence into anchor/positive/negative triplets.
# The positive target matches the annotation; opposing and NEUTRAL targets become negatives. Duplicate triplets are removed before shuffling.
# Build PRO / CON / NEUTRAL training triplets
triplets = []
for _, row in balanced_train_df.iterrows():
    sentence = str(row["sentence"]).strip()
    direction = row["leakage_direction"]
    if not sentence:
        continue
    if direction == "PRO":
        for i in range(4):
            triplets.append({
                "anchor": sentence,
                "positive": pro_targets[i],
                "negative": con_targets[i]
            })
            triplets.append({
                "anchor": sentence,
                "positive": pro_targets[i],
                "negative": neutral_targets[i]
            })
    elif direction == "CON":
        for i in range(4):
            triplets.append({
                "anchor": sentence,
                "positive": con_targets[i],
                "negative": pro_targets[i]
            })
            triplets.append({
                "anchor": sentence,
                "positive": con_targets[i],
                "negative": neutral_targets[i]
            })
    elif direction == "NEUTRAL":
        for i in range(4):
            triplets.append({
                "anchor": sentence,
                "positive": neutral_targets[i],
                "negative": pro_targets[i]
            })
            triplets.append({
                "anchor": sentence,
                "positive": neutral_targets[i],
                "negative": con_targets[i]
            })
triplets_df = pd.DataFrame(triplets)
triplets_df = (
    triplets_df
    .drop_duplicates()
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
print("Total triplets:", len(triplets_df))
triplets_df.head()


Total triplets: 1736


,anchor,positive,negative
0,You also needed to engage with the childcare b...,Neither side is clearly favored by this feedback.,The judge favors Pro over Con.
1,You also struggled to respond to PRO's point a...,The judge favors Pro over Con.,The judge favors Con over Pro.
2,When the pro side argued that subsidies cannot...,This feedback does not indicate which side won.,Con wins the debate.
3,"Pro provided a clearer, empirically backed mec...",The judge favors Pro over Con.,The judge favors Con over Pro.
4,"The crossfire exchanges were close, but PRO's ...",Pro has the stronger argument than Con.,This is general feedback rather than a ballot ...


In [11]:
# EXPERIMENT 2 — Verify the intended comparison count without changing training rows.
# Each real sentence contributes four formulations against each of two incorrect classes: eight triplets total.
expected_triplets = 8 * balanced_train_df["sentence"].astype(str).str.strip().ne("").sum()
print("Real training sentences:", len(balanced_train_df))
print("Expected comparisons before exact-triplet deduplication:", expected_triplets)
print("Training triplets after the historical deduplication step:", len(triplets_df))


Real training sentences: 217
Expected comparisons before exact-triplet deduplication: 1736
Training triplets after the historical deduplication step: 1736


In [12]:
# STEP 5 — Load a fresh BGE checkpoint for Experiment 2.
# Starting fresh prevents weights or state from the baseline analysis from carrying into this training run.
# Reset to a completely fresh BGE for Experiment 2
from sentence_transformers import SentenceTransformer

finetune_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("Fresh BGE loaded for Experiment 2.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Fresh BGE loaded for Experiment 2.


In [13]:
# STEP 6 — Manually inspect a small triplet sample before training.
# This quality-control step helps catch reversed positives/negatives or malformed ballot sentences.
# Inspect training examples manually
for i in range(min(10, len(triplets_df))):
    row = triplets_df.iloc[i]
    print("\n" + "=" * 80)
    print("ANCHOR:")
    print(row["anchor"])
    print("\nPOSITIVE:")
    print(row["positive"])
    print("\nNEGATIVE:")
    print(row["negative"])



ANCHOR:
You also needed to engage with the childcare burden argument because many people may not avoid childbirth because they lack money alone, but because they fear things like career interruption, lack of childcare availability, and how the responsibility falling mainly on women.

POSITIVE:
Neither side is clearly favored by this feedback.

NEGATIVE:
The judge favors Pro over Con.

ANCHOR:
You also struggled to respond to PRO's point about rural areas with no childcare options.

POSITIVE:
The judge favors Pro over Con.

NEGATIVE:
The judge favors Con over Pro.

ANCHOR:
When the pro side argued that subsidies cannot cover the costs of raising a child, I find this evidence pretty compelling.

POSITIVE:
This feedback does not indicate which side won.

NEGATIVE:
Con wins the debate.

ANCHOR:
Pro provided a clearer, empirically backed mechanism directly linking childcare to higher birth probabilities.

POSITIVE:
The judge favors Pro over Con.

NEGATIVE:
The judge favors Con over Pro.

A

In [14]:
# STEP 7 — Package the triplets and define the training objective.
# TripletLoss pulls each anchor toward its positive target and pushes it away from the negative target.
# Cell 42 — Prepare Hugging Face Dataset + TripletLoss
from datasets import Dataset
from sentence_transformers import losses
train_dataset = Dataset.from_pandas(
    triplets_df[
        ["anchor", "positive", "negative"]
    ],
    preserve_index=False
)
train_loss = losses.TripletLoss(
    model=finetune_model
)
print(train_dataset)


Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 1736
})


/tmp/ipykernel_2369/1465893195.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses


## Training

Train the fresh encoder with the original Experiment 2 TripletLoss settings.


In [15]:
# STEP 8 — Configure the SentenceTransformers trainer.
# Two epochs, batch size 16, learning rate 2e-5, 10% warmup, and seed 42 are fixed for reproducibility.
# Cell 43 — Training configuration

from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments
)

training_args = SentenceTransformerTrainingArguments(
    output_dir="/content/bge-ballot-experiment",

    num_train_epochs=2,

    per_device_train_batch_size=16,

    learning_rate=2e-5,

    warmup_ratio=0.1,

    seed=42,

    logging_steps=5,

    save_strategy="epoch",

    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=finetune_model,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss
)

print("Trainer ready.")


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Trainer ready.


In [16]:
# STEP 9 — Fine-tune the BGE encoder on the generated triplets.
# This updates model weights; no VALIDATION or TEST rows are passed to trainer.train().
# Cell 44 — Fine-tune BGE

trainer.train()


Step,Training Loss
5,4.992635
10,4.986868
15,4.978899
20,4.979796
25,4.951722
30,4.917474
35,4.886698
40,4.876408
45,4.745343
50,4.677655


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=218, training_loss=4.3613708434848615, metrics={'train_runtime': 77.3169, 'train_samples_per_second': 44.906, 'train_steps_per_second': 2.82, 'total_flos': 0.0, 'train_loss': 4.3613708434848615, 'epoch': 2.0})

In [17]:
# STEP 10 — Save the completed Experiment 2 encoder.
# The dedicated path keeps this model separate from the untouched original BGE checkpoint.
# Cell 45 — Save fine-tuned model

MODEL_PATH = "/content/bge-ballot-finetuned-experiment"

finetune_model.save(
    MODEL_PATH
)

print("Saved fine-tuned model to:")
print(MODEL_PATH)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model to:
/content/bge-ballot-finetuned-experiment


## Validation Evaluation

Use direct PRO/CON/NEUTRAL argmax inference for every final Experiment 2 metric.


In [18]:
# STEP 19 — Encode the PRO, CON, and NEUTRAL target banks with the fine-tuned encoder.
# These vectors define the three semantic reference regions used by the later argmax classifier.
pro_embeddings_ft = finetune_model.encode(
    pro_targets,
    normalize_embeddings=True
)

con_embeddings_ft = finetune_model.encode(
    con_targets,
    normalize_embeddings=True
)

neutral_embeddings_ft = finetune_model.encode(
    neutral_targets,
    normalize_embeddings=True
)


In [19]:
# STEP 20 — Encode every VALIDATION sentence with the same fine-tuned encoder.
# No model weights are updated here; this is inference on the held-out validation rows.
# Encode VALIDATION sentences with the fine-tuned model

validation_sentences = (
    validation_df["sentence"]
    .fillna("")
    .astype(str)
    .tolist()
)

ft_sentence_embeddings = finetune_model.encode(
    validation_sentences,
    normalize_embeddings=True
)

print(ft_sentence_embeddings.shape)


(61, 768)


In [20]:
# STEP 21 — Calculate each sentence’s best similarity to every target bank.
# max(axis=1) keeps the strongest matching prompt within PRO, CON, and NEUTRAL.
pro_sim = ft_sentence_embeddings @ pro_embeddings_ft.T
con_sim = ft_sentence_embeddings @ con_embeddings_ft.T
neutral_sim = ft_sentence_embeddings @ neutral_embeddings_ft.T

validation_df["ft_pro_score"] = pro_sim.max(axis=1)
validation_df["ft_con_score"] = con_sim.max(axis=1)
validation_df["ft_neutral_score"] = neutral_sim.max(axis=1)


In [21]:
# STEP 22 — Make a direct three-way prediction with argmax.
# The label attached to the largest PRO/CON/NEUTRAL similarity becomes the predicted direction; no threshold is used here.
import numpy as np
scores = np.column_stack([
    validation_df["ft_pro_score"],
    validation_df["ft_con_score"],
    validation_df["ft_neutral_score"]
])
labels = np.array([
    "PRO",
    "CON",
    "NEUTRAL"
])
validation_df["ft_3way_prediction"] = labels[
    np.argmax(scores, axis=1)
]


In [22]:
# EXPERIMENT 2 — Compute all final metrics from direct three-way argmax predictions.
# Accuracy, Macro-F1, leakage-only accuracy, the report, and the confusion matrix use ft_3way_prediction consistently.
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

y_true_exp2 = validation_df["leakage_direction"]
y_pred_exp2 = validation_df["ft_3way_prediction"]
exp2_accuracy = accuracy_score(y_true_exp2, y_pred_exp2)
exp2_macro_f1 = f1_score(y_true_exp2, y_pred_exp2, average="macro", zero_division=0)
exp2_leakage_mask = y_true_exp2.isin(["PRO", "CON"])
exp2_leakage_accuracy = accuracy_score(
    y_true_exp2[exp2_leakage_mask],
    y_pred_exp2[exp2_leakage_mask]
)

print(f"Experiment 2 accuracy: {exp2_accuracy:.4f}")
print(f"Experiment 2 Macro-F1: {exp2_macro_f1:.4f}")
print(f"Experiment 2 leakage-only direction accuracy: {exp2_leakage_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(
    y_true_exp2,
    y_pred_exp2,
    labels=["CON", "NEUTRAL", "PRO"],
    digits=3,
    zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted; CON, NEUTRAL, PRO):")
print(confusion_matrix(y_true_exp2, y_pred_exp2, labels=["CON", "NEUTRAL", "PRO"]))


Experiment 2 accuracy: 0.7049
Experiment 2 Macro-F1: 0.6892
Experiment 2 leakage-only direction accuracy: 0.6970

Classification report:
              precision    recall  f1-score   support

         CON      0.583     0.737     0.651        19
     NEUTRAL      0.952     0.714     0.816        28
         PRO      0.562     0.643     0.600        14

    accuracy                          0.705        61
   macro avg      0.699     0.698     0.689        61
weighted avg      0.748     0.705     0.715        61

Confusion matrix (rows=true, columns=predicted; CON, NEUTRAL, PRO):
[[14  0  5]
 [ 6 20  2]
 [ 4  1  9]]


## Error Analysis

Inspect three-way validation errors and missed PRO examples.


### Misclassified Examples

The following examples examine where the three-way model disagrees with the annotated validation labels. Particular attention is given to confusion between PRO and CON, directional examples predicted as NEUTRAL, and NEUTRAL examples incorrectly assigned a direction.

In [23]:
# STEP 24 — Inspect all incorrect three-way predictions.
# score_margin measures confidence: small gaps between the top two class scores identify ambiguous errors.
cols = [
    "sentence",
    "leakage_direction",
    "ft_3way_prediction",
    "ft_pro_score",
    "ft_con_score",
    "ft_neutral_score",
    "section",
    "leakage_strength"
]

errors = validation_df[
    validation_df["leakage_direction"]
    != validation_df["ft_3way_prediction"]
][cols].copy()

errors["score_margin"] = (
    errors[
        ["ft_pro_score", "ft_con_score", "ft_neutral_score"]
    ].max(axis=1)
    -
    errors[
        ["ft_pro_score", "ft_con_score", "ft_neutral_score"]
    ].apply(
        lambda x: sorted(x, reverse=True)[1],
        axis=1
    )
)

errors.sort_values("score_margin")


,sentence,leakage_direction,ft_3way_prediction,ft_pro_score,ft_con_score,ft_neutral_score,section,leakage_strength,score_margin
10,"Most importantly, CON directly challenged PRO’...",CON,PRO,0.999123,0.999112,-0.954633,Reason for Decision,2,0.000012
36,"On the Productivity clash, the pro side clearl...",NEUTRAL,CON,0.995337,0.995351,-0.951631,Reason for Decision,0,0.000014
9,PRO explained well why childcare can reduce pr...,CON,PRO,0.995991,0.995938,-0.952430,Reason for Decision,1,0.000053
17,"CON’s Final Focus on cost, choice, and speed w...",PRO,CON,0.998941,0.999032,-0.954789,Reason for Decision,2,0.000091
27,Regarding Pro’s proposed solution—using AI to ...,CON,PRO,0.998270,0.998107,-0.953493,Reason for Decision,1,0.000163
22,Offered rebuttals addressing Pro’s assumptions...,NEUTRAL,PRO,0.996388,0.996204,-0.949521,Comments to CON,0,0.000185
60,The CON side could not prove how the currect s...,PRO,CON,0.998403,0.998817,-0.954530,Reason for Decision,1,0.000414
48,"They also controlled the entire debate, clearl...",CON,PRO,0.994982,0.994509,-0.950267,Reason for Decision,2,0.000473
1,"Con2 also claims that STEMS are wasting money,...",PRO,CON,0.997668,0.998153,-0.954126,Reason for Decision,1,0.000485
55,"In second crossfire, don't ask how the PRO arg...",NEUTRAL,PRO,0.996324,0.995823,-0.951579,Comments to CON,0,0.000501


In [24]:
# STEP 25 — Isolate missed PRO examples for targeted error analysis.
# This diagnostic reveals whether PRO language is being mistaken for CON or NEUTRAL.
pro_errors = validation_df[
    (validation_df["leakage_direction"] == "PRO")
    &
    (validation_df["ft_3way_prediction"] != "PRO")
][cols]

pro_errors


,sentence,leakage_direction,ft_3way_prediction,ft_pro_score,ft_con_score,ft_neutral_score,section,leakage_strength
1,"Con2 also claims that STEMS are wasting money,...",PRO,CON,0.997668,0.998153,-0.954126,Reason for Decision,1
17,"CON’s Final Focus on cost, choice, and speed w...",PRO,CON,0.998941,0.999032,-0.954789,Reason for Decision,2
33,"Also, prepare short and precise questions in t...",PRO,NEUTRAL,-0.944580,-0.945343,0.995643,Comments to CON,1
37,The con side failed to fully rebut this.,PRO,CON,0.998278,0.999087,-0.953094,Reason for Decision,1
60,The CON side could not prove how the currect s...,PRO,CON,0.998403,0.998817,-0.954530,Reason for Decision,1


## Experiment Summary

- **Data:** all PRO and CON TRAIN rows plus 60 sampled NEUTRAL rows (`random_state=42`).
- **Training:** eight triplet comparisons per real sentence, TripletLoss, 2 epochs, batch size 16, learning rate 2e-5, warmup ratio 0.1, seed 42.
- **Inference:** direct maximum-similarity argmax over PRO, CON, and NEUTRAL target banks.
- **Metrics:** accuracy, Macro-F1, leakage-only direction accuracy, classification report, and confusion matrix all use the same `ft_3way_prediction` column.
- **TEST:** untouched and not loaded.

The older threshold-based result is intentionally excluded because it is not comparable to the final direct three-way evaluation. No superiority claim is made.


## Experiment 2 Summary

Explicitly introducing NEUTRAL as a third semantic target improved three-way classification compared with directional-only fine-tuning. On VALIDATION, the model achieved **70.45% accuracy**, **0.6250 Macro-F1**, and **61.11% leakage-only direction accuracy**.

The results suggest that explicitly modeling NEUTRAL addresses an important limitation of Experiment 1. However, PRO/CON confusion remains, indicating that learning the distinction between the two leakage directions is still difficult.

Experiment 3 keeps the same three-way training method but removes the 60-example NEUTRAL subsampling and uses the full TRAIN distribution.